In [ ]:
import json
import os
import google.generativeai as genai
import time
from google.api_core.exceptions import ResourceExhausted
import typing
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GENAI_API_KEY_1")
if not api_key:
    raise ValueError("API key not found. Please set the GENAI_API_KEY environment variable.")

class KeywordList(typing.TypedDict):
    keyword_list: list[str]

genai.configure(api_key=api_key)
system_instruction="You are a conversational AI assistant. You are asked to generate a few keywords that summarize the content of a given text. Only give the keywords in csv format and in lowercase."
model = genai.GenerativeModel(model_name="gemini-1.5-flash-8b", system_instruction=system_instruction)
generation_config = genai.GenerationConfig(
    max_output_tokens=200,
    temperature=0.1,
    candidate_count=1,
    response_mime_type="application/json",
    response_schema=KeywordList,
)

def generate_keywords(text):
    while True:
        try:
            model_output = model.generate_content(contents=text, generation_config=generation_config)
            keywords = model_output.candidates[0].content.parts[0].text
            keywords = json.loads(keywords)["keyword_list"]
            keywords = ",".join([keyword.lower() for keyword in keywords])
            print(keywords)
            return keywords
        except ResourceExhausted:
            print("Rate limit exceeded. Waiting for 5 seconds...")
            time.sleep(5)

In [ ]:
text = "[The episode begins in Gumball and Darwin's room, where Darwin is seen typing on a laptop. The camera then zooms to his laptop's screen, showing Darwin moving the letter \"n\" towards Alan's name to finish making his name in the yearbook slot. Carrie then starts texting Darwin]\nCarrie: [texting] Hey Darwin, how's that yearbook coming along?\nDarwin: [texting] I'm just about to add Alan.\nCarrie: [texting] Are you sure there's enough space for his ego?"

keywords = generate_keywords(text)

yearbook,darwin,alan,texting,ego


In [18]:
import json
import unicodedata
import re

with open("transcripts.jsonl", "r", encoding="utf-8") as f:
    transcripts = [json.loads(line) for line in f]

for transcript in transcripts:
    # Normalize and decode the text
    text = unicodedata.normalize("NFKD", transcript["text"]).encode('utf-8').decode('utf-8')
    # Remove non-ASCII characters and spaces before or after them
    text = re.sub(r'\s?[^\x00-\x7F]|\s?[^\x00-\x7F]\s', '', text)
    transcript["text"] = text

# Rewrite the file with the new text
with open("test.jsonl", "w", encoding="utf-8") as f:
    for transcript in transcripts:
        f.write(json.dumps(transcript, ensure_ascii=False) + "\n")